In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from torchvision import datasets, transforms

import wandb

# Пример простой сети
class Net(nn.Module):
    def __init__(self, hidden_size, dropout, activation):
        super().__init__()
        self.fc1 = nn.Linear(32*32*3, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.act = getattr(torch.nn, activation)()
        self.fc2 = nn.Linear(hidden_size, 10)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.act(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Пример функций обучения и валидации
def train(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for data, targets in loader:
        data, targets = data.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * data.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(targets).sum().item()
        total += targets.size(0)
    return running_loss / total, correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for data, targets in loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * data.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(targets).sum().item()
            total += targets.size(0)
    return running_loss / total, correct / total

# Основная функция для sweep
def main_wandb(config=None):
    with wandb.init(config=config):
        config = wandb.config
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        transform = transforms.ToTensor()
        train_set = datasets.CIFAR10(root='.', train=True, download=True, transform=transform)
        val_set = datasets.CIFAR10(root='.', train=False, download=True, transform=transform)
        train_loader = DataLoader(train_set, batch_size=config['batch_size'], shuffle=True)
        val_loader = DataLoader(val_set, batch_size=config['batch_size'])

        model = Net(config['hidden_size'], config['dropout'], config['activation']).to(device)
        optimizer = optim.Adam(model.parameters(), lr=config['lr'])
        criterion = nn.CrossEntropyLoss()

        best_val_acc = 0.0
        for epoch in range(config.epochs):
            train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
            val_loss, val_acc = validate(model, val_loader, criterion, device)
            wandb.log({
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "epoch": epoch
            })
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), "best_model.pt")

# Пример sweep config
sweep_config = {
    'method': 'bayes',  # можно grid, random, bayes
    'metric': {'name': 'val_acc', 'goal': 'maximize'},
    'parameters': {
        'lr': {
            'values': [0.1, 0.01, 0.001]
        },
        'batch_size': {
            'values': [64, 128]
        },
        'hidden_size': {
            'values': [128, 256]
        },
        'dropout': {
            'min': 0.1,
            'max': 0.5
        },
        'activation': {
            'values': ['ReLU', 'Tanh']
        },
        'epochs': {
            'value': 3
        }
    }
}

In [2]:
# Авторизация в W&B
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
my_secret = user_secrets.get_secret("wandb_api_key")
wandb.login(key=my_secret)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ladaogneva13 (ladaogneva13-bauman-moscow-state-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
# Исправьте на имя вашего логина, задайте имя проекта
sweep_id = wandb.sweep(sweep_config, entity='kvsbmstu', project='cifar10-classification-sweep-10')
# Запускаем поиск гиперпараметров
wandb.agent(sweep_id, function=main_wandb, count=10)

Create sweep with ID: cilvu8jb
Sweep URL: https://wandb.ai/ladaogneva13-bauman-moscow-state-technical-university/cifar10-classification-sweep-10/sweeps/cilvu8jb


wandb: Agent Starting Run: xubwb9re with config:
wandb: 	activation: ReLU
wandb: 	batch_size: 128
wandb: 	dropout: 0.4452839776995493
wandb: 	epochs: 3
wandb: 	hidden_size: 128
wandb: 	lr: 0.1


100%|██████████| 170M/170M [00:05<00:00, 28.6MB/s] 


epoch,▁▅█
train_acc,▄▁█
train_loss,█▁▁
val_acc,▁▁▁
val_loss,▂▁█
epoch,2
train_acc,0.09986
train_loss,2.30807
val_acc,0.1
val_loss,2.30813


wandb: Agent Starting Run: 8hj6mw23 with config:
wandb: 	activation: Tanh
wandb: 	batch_size: 64
wandb: 	dropout: 0.27807644050327035
wandb: 	epochs: 3
wandb: 	hidden_size: 256
wandb: 	lr: 0.1


...

epoch,▁▅█
train_acc,▁▆█
train_loss,█▁▁
val_acc,▁██
val_loss,█▁█
epoch,2
train_acc,0.17546
train_loss,2.12531
val_acc,0.2182
val_loss,2.04236


wandb: Agent Starting Run: tstnv33x with config:
wandb: 	activation: ReLU
wandb: 	batch_size: 64
wandb: 	dropout: 0.4258899580640946
wandb: 	epochs: 3
wandb: 	hidden_size: 128
wandb: 	lr: 0.01


epoch,▁▅█
train_acc,▁█▄
train_loss,█▁▁
val_acc,█▁▇
val_loss,▁█▃
epoch,2
train_acc,0.13508
train_loss,2.22576
val_acc,0.1593
val_loss,2.19589


wandb: Agent Starting Run: ykzui2y4 with config:
wandb: 	activation: ReLU
wandb: 	batch_size: 64
wandb: 	dropout: 0.22293931649030607
wandb: 	epochs: 3
wandb: 	hidden_size: 128
wandb: 	lr: 0.001


epoch,▁▅█
train_acc,▁▆█
train_loss,█▃▁
val_acc,▁▁█
val_loss,█▄▁
epoch,2
train_acc,0.34314
train_loss,1.81798
val_acc,0.3829
val_loss,1.72997
